# Ch01-04: 이상치 탐지 - 기계학습 기반


## 학습 목표

- Isolation Forest의 분리 깊이와 이상 점수의 관계를 이해한다
- LOF의 지역 밀도 비율 계산 원리를 파악한다
- One-Class SVM의 커널 기반 경계 학습을 이해한다
- 앙상블 방법으로 이상치 탐지의 안정성을 높인다


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 이론적 배경


### 1. Isolation Forest (Liu et al., 2008)

이상 점수: $s(x,n) = 2^{-E[h(x)]/c(n)}$

$c(n) = 2H(n-1)-2(n-1)/n$, $H(i)=\ln(i)+\gamma$

### 2. LOF (Breunig et al., 2000)

$$\text{LOF}_k(x) = \frac{1}{|N_k(x)|}\sum_{o\in N_k(x)}\frac{\text{lrd}_k(o)}{\text{lrd}_k(x)}$$

### 3. One-Class SVM

$$\min_{w,\xi,\rho}\frac{1}{2}\|w\|^2+\frac{1}{\nu n}\sum\xi_i-\rho$$
$$\text{s.t.}\; w\cdot\Phi(x_i)\geq\rho-\xi_i$$


## 구현 가이드

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X_n = np.random.multivariate_normal([0,0], [[1,0.5],[0.5,1]], 300)
X_o = np.vstack([np.random.uniform(-5,5,(15,2)), np.random.multivariate_normal([4,-3],[[0.1,0],[0,0.1]],5)])
X = np.vstack([X_n,X_o]); y = np.array([1]*300+[-1]*20)
Xs = StandardScaler().fit_transform(X)

models = [('IF', IsolationForest(n_estimators=100, contamination=0.06, random_state=42)),
          ('LOF', LocalOutlierFactor(n_neighbors=20, contamination=0.06)),
          ('OCSVM', OneClassSVM(kernel='rbf', gamma='scale', nu=0.06))]

fig, axes = plt.subplots(1,3,figsize=(18,5))
for ax, (name, m) in zip(axes, models):
    pred = m.fit_predict(Xs)
    ax.scatter(X[pred==1,0],X[pred==1,1],c='blue',s=15,alpha=0.5,label='정상')
    ax.scatter(X[pred==-1,0],X[pred==-1,1],c='red',s=40,marker='x',label='이상치')
    ax.set_title(name); ax.legend()
plt.tight_layout(); plt.show()


---
## 연습 문제

### 문제 1 [★]

IF/LOF/OCSVM의 하이퍼파라미터 민감도를 분석하라. 각 설정에서 F1을 계산하고 시각화.

In [ ]:
# TODO: 그리드 탐색 + F1 비교


### 문제 2 [★★]

Isolation Forest를 직접 구현하라 (sklearn 금지). iTree/iForest 클래스, 이상점수 계산, sklearn과 AUC 비교.

> **힌트**: 서브샘플 ψ=256, 깊이 제한=ceil(log2(ψ)).

In [ ]:
class IsolationTree:
    pass
class IsolationForestCustom:
    pass


### 문제 3 [★★]

LOF 직접 구현. 균일분포에서 LOF≈1 확인. 밀도 변동 데이터(10:1)에서 LOF 장점 시연.

In [ ]:
def compute_lof(X, k=20):
    # TODO
    pass


### 문제 4 [★★★]

이상치 탐지 앙상블 설계.
1) IF/LOF/OCSVM 점수 정규화
2) 평균/최대/가중평균/다수결 결합
3) 5가지 시나리오에서 단일 vs 앙상블 강건성 평가

In [ ]:
# TODO: 앙상블 프레임워크
# TODO: 시나리오별 실험


---
## 참고 자료

- Liu, Ting & Zhou (2008). Isolation Forest. ICDM.
- Breunig et al. (2000). LOF. ACM SIGMOD.
- Scholkopf et al. (2001). Support of High-Dim Distribution.
